<a href="https://colab.research.google.com/github/TarfaMajeed/Projects/blob/main/Object_detection_on_Satellite_Images_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/Roads /content/

import torch
import torchvision
from torchvision import transforms
import cv2
import numpy as np
from matplotlib import pyplot as plt
import os
from sklearn.metrics import precision_score, recall_score, f1_score


model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights='COCO_V1')
model.eval()

transform = transforms.Compose([transforms.ToTensor()])

# Vehicle classes in COCO
VEHICLE_IDS = [3, 8, 6]  # car, truck, bus

# Accumulators for vehicle detection statistics
total_detections = 0
total_images = 0

def overlay_vehicle_masks(image, output, score_thresh=0.15, mask_thresh=0.4, alpha=0.5):
    overlay = image.copy()
    detections = 0
    for idx, label in enumerate(output['labels']):
        if label.item() in VEHICLE_IDS and output['scores'][idx] > score_thresh:
            detections += 1
            mask = output['masks'][idx, 0].cpu().numpy()
            overlay[mask > mask_thresh] = overlay[mask > mask_thresh] * (1 - alpha) + np.array([255, 0, 0]) * alpha  # Blue
    if detections > 0:
        print(f"   → Applied {detections} vehicle masks (blue)")
    return overlay, detections

def overlay_roads(image, road_mask, alpha=0.5):
    overlay = image.copy()
    overlay[road_mask > 0] = overlay[road_mask > 0] * (1 - alpha) + np.array([0, 255, 255]) * alpha
    return overlay

def approximate_buildings(image, road_mask=None, alpha=0.5):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_OTSU)
    kernel = np.ones((12,12), np.uint8)
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=3)
    if road_mask is not None:
        closed[road_mask > 0] = 0
    overlay = image.copy()
    overlay[closed > 0] = overlay[closed > 0] * (1 - alpha) + np.array([0, 0, 255]) * alpha  # Red
    return overlay

def load_road_mask(mask_path):
    if os.path.exists(mask_path):
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        return (mask > 127).astype(np.uint8)
    return np.zeros((600, 600), dtype=np.uint8)

def approximate_roads(image, alpha=0.5):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    road_approx = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    kernel = np.ones((9,9), np.uint8)
    road_approx = cv2.morphologyEx(road_approx, cv2.MORPH_OPEN, kernel)
    overlay = image.copy()
    overlay[road_approx > 0] = overlay[road_approx > 0] * (1 - alpha) + np.array([0, 255, 255]) * alpha
    return overlay, road_approx

# Process ALL training images
print("Processing ALL training images...\n")
images_path = '/content/Roads/training/images'
masks_path = '/content/Roads/training/groundtruth'
image_files = sorted(os.listdir(images_path))

for img_name in image_files:
    total_images += 1
    img_path = os.path.join(images_path, img_name)
    image_bgr = cv2.imread(img_path)
    if image_bgr is None:
        print(f"Failed to load {img_path}")
        continue
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    print(f"Processing training image: {img_name}")

    # Groundtruth roads (yellow)
    road_mask_path = os.path.join(masks_path, img_name)
    road_mask = load_road_mask(road_mask_path)
    overlay = overlay_roads(image_bgr, road_mask)

    # Vehicle detection (blue masks only)
    input_tensor = transform(image_rgb).unsqueeze(0)
    with torch.no_grad():
        output = model(input_tensor)[0]
    overlay, detections = overlay_vehicle_masks(overlay, output, score_thresh=0.15)
    total_detections += detections

    # Approximate buildings (red)
    final = approximate_buildings(overlay, road_mask)

    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
    plt.title(f'{img_name}\nYellow: Roads (GT) | Blue: Vehicles | Red: Approx Buildings')
    plt.axis('off')
    plt.show()

# Process ALL test images
print("\nProcessing ALL test images (no GT roads)...\n")
test_path = '/content/Roads/test_set_images'
for i in range(1, 51):
    subdir = f'test_{i}'
    folder_path = os.path.join(test_path, subdir)
    img_name = f'{subdir}.png'
    img_path = os.path.join(folder_path, img_name)

    if not os.path.exists(img_path):
        print(f"Skipped {img_path} - not found")
        continue

    total_images += 1  # Count test images too for average
    image_bgr = cv2.imread(img_path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    print(f"Processing test image: {subdir}")

    # Approximate roads (yellow)
    overlay, approx_road_mask = approximate_roads(image_bgr)

    # Vehicles
    input_tensor = transform(image_rgb).unsqueeze(0)
    with torch.no_grad():
        output = model(input_tensor)[0]
    overlay, detections = overlay_vehicle_masks(overlay, output, score_thresh=0.15)
    total_detections += detections

    # Buildings
    final = approximate_buildings(overlay, approx_road_mask)

    # Visualize
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(final, cv2.COLOR_BGR2RGB))
    plt.title(f'{subdir}\nYellow: Approx Roads | Blue: Vehicles | Red: Approx Buildings')
    plt.axis('off')
    plt.show()

# Final Summary and Metrics
print("\n" + "="*60)
print("FINAL EVALUATION SUMMARY")
print("="*60)
print(f"Total images processed: {total_images}")
print(f"Total vehicle instances detected: {total_detections}")
print(f"Average vehicles detected per image: {total_detections / total_images:.2f}")

print("\n--- Metric Explanation ---")
print("Road Segmentation (Training images only):")
print("   → Using GROUNDTRUTH masks → Precision = 1.00, Recall = 1.00, F1 = 1.00, IoU = 1.00")

print("\nVehicle Detection (Cars/Trucks/Buses):")
print("   → No groundtruth annotations available in this dataset")
print("   → Cannot compute true Precision, Recall, F1, Accuracy, or IoU")
print("   → Only detection count reported (with low threshold 0.15 to maximize recall)")
print("   → These detections may include false positives due to top-down view mismatch")

print("\nBuilding Segmentation:")
print("   → Pure heuristic (thresholding + morphology)")
print("   → No groundtruth → No quantitative metrics possible")

print("\nThis dataset is designed ONLY for road extraction evaluation.")
print("Standard road segmentation metrics (if you had predicted roads):")
print("   Precision, Recall, F1-score, IoU (Jaccard) at pixel level")
print("But here roads are perfect (GT), vehicles/buildings are bonus approximations.")

print("\nDone! Scroll up to see all visualized results.")